In [ ]:
import os

GPU = 0
DATASET = "cifar10"
SEEDS = [42, 43, 44]
SCALES_TO_RUN = ["small", "deeper-small", "medium", "large", "vlarge"]
SMOKE = False

os.environ["CUDA_VISIBLE_DEVICES"] = str(GPU)
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

ROOT = os.getcwd()
while not os.path.exists(os.path.join(ROOT, "pyproject.toml")):
    ROOT = os.path.dirname(ROOT)

CKPT = os.path.join(ROOT, "final-checkpoints", "smoke" if SMOKE else "")
JSONL = os.path.join(ROOT, "final-checkpoints", "results.jsonl")
os.makedirs(os.path.join(CKPT, DATASET), exist_ok=True)

In [ ]:
import sys, time, json, pickle, gc
from collections import OrderedDict
import numpy as np
import jax, jax.numpy as jnp, optax

sys.path.insert(0, ROOT)
from pst_dtlgn.data.pipeline import BinaryPipeline, TernaryPipeline, load_cifar10, load_cifar100
from pst_dtlgn.network.topology import random_sparse
from pst_dtlgn.network.network import PolynomialNetwork
from pst_dtlgn.network.harden import harden_network_fast, TernaryLearnedCircuit
from pst_dtlgn.core.gate_library import GateLibrary
from pst_dtlgn.binary_baseline import BinaryDLGN, GroupSum, harden_binary_network, BinaryLearnedCircuit, BIN_TRUTH_TABLES
from pst_dtlgn.training.trainer import train, make_ternary_loss
from pst_dtlgn.training.binary_trainer import train_binary
from pst_dtlgn.analysis.evaluation import evaluate_soft

print("jax devices:", jax.devices())

In [ ]:
BINARY_LR = 0.01
LAMBDA_MAX = 0.1
LAMBDA_GAMMA = 2.0
GROUPSUM_TAU = 33.3
BATCH_SIZE = 100
RESOLUTION = 4

FULL_SCALES = OrderedDict([
    ("small",        {"widths": [12000] * 4,  "ternary_lr": 0.003, "steps": 100_000}),
    ("deeper-small", {"widths": [12000] * 5,  "ternary_lr": 0.003, "steps": 100_000}),
    ("medium",       {"widths": [24000] * 4,  "ternary_lr": 0.001, "steps": 150_000}),
    ("large",        {"widths": [36000] * 4,  "ternary_lr": 0.001, "steps": 200_000}),
    ("vlarge",       {"widths": [48000] * 4,  "ternary_lr": 0.001, "steps": 150_000}),
    ("huge",         {"widths": [128000] * 4, "ternary_lr": 0.001, "steps": 200_000}),
    ("vhuge",        {"widths": [256000] * 4, "ternary_lr": 0.001, "steps": 250_000}),
])

if SMOKE:
    SCALES = OrderedDict([("smoke", {"widths": [2000] * 2, "ternary_lr": 0.003, "steps": 200})])
    SEEDS = [42]
else:
    SCALES = OrderedDict((k, FULL_SCALES[k]) for k in SCALES_TO_RUN)

N_CLASSES = 10 if DATASET == "cifar10" else 100
GROUPSUM_K = N_CLASSES
ARCH_LABEL = {"ternary": "TLGN", "binary": "BLGN"}
GPU_NAME = jax.devices()[0].device_kind
print(DATASET, "| scales:", list(SCALES), "| seeds:", SEEDS)

In [ ]:
loader = load_cifar10 if DATASET == "cifar10" else load_cifar100
data = loader(os.path.join(ROOT, "data", DATASET))

bin_pipe, ter_pipe = BinaryPipeline(resolution=RESOLUTION), TernaryPipeline(resolution=RESOLUTION)
bin_train = bin_pipe.fit_transform(data["train_x"], mode="hard")
bin_test = bin_pipe.transform(data["test_x"], mode="hard")
ter_train = ter_pipe.fit_transform(data["train_x"], mode="hard")
ter_test = ter_pipe.transform(data["test_x"], mode="hard")
INPUT_DIM = bin_train.shape[1]

bin_train_j, ter_train_j = jnp.array(bin_train), jnp.array(ter_train)
bin_test_j, ter_test_j = jnp.array(bin_test), jnp.array(ter_test)
train_y_j = jnp.array(data["train_y"])
train_y, test_y = data["train_y"], data["test_y"]
print("train", data["train_x"].shape, "test", data["test_x"].shape, "encoded dim", INPUT_DIM)

In [ ]:
def ternary_forward(circuit, X):
    h = X.astype(np.int8)
    for tt, conn in zip(circuit.truth_tables, circuit.connections):
        a = h[:, conn[:, 0]].astype(np.int32)
        b = h[:, conn[:, 1]].astype(np.int32)
        gather = np.arange(tt.shape[0], dtype=np.int64)[None, :] * 9 + (a + 1) * 3 + (b + 1)
        h = tt.reshape(-1)[gather].astype(np.int8)
    return h

def binary_forward(circuit, X):
    h = X.astype(np.int8)
    bt = np.asarray(BIN_TRUTH_TABLES).reshape(-1)
    for gidx, conn in zip(circuit.gate_indices, circuit.connections):
        gidx = np.asarray(gidx, dtype=np.int64)
        a = h[:, conn[:, 0]].astype(np.int32)
        b = h[:, conn[:, 1]].astype(np.int32)
        h = bt[gidx[None, :] * 4 + (a * 2 + b)].astype(np.int8)
    return h

def circuit_test_metrics(circuit, x_np, y, n_classes, arch, chunk=2000):
    preds = np.empty(len(x_np), dtype=np.int32)
    zeros = total = 0
    fwd = ternary_forward if arch == "ternary" else binary_forward
    for s0 in range(0, len(x_np), chunk):
        s1 = min(s0 + chunk, len(x_np))
        out = fwd(circuit, x_np[s0:s1])
        g = out.shape[1] // n_classes
        preds[s0:s1] = out.reshape(out.shape[0], n_classes, g).sum(-1).argmax(-1)
        if arch == "ternary":
            zeros += int((out == 0).sum())
        total += out.size
    unk = zeros / total if (arch == "ternary" and total) else 0.0
    return float(np.mean(preds == y)), unk

In [ ]:
def read_rows():
    if not os.path.exists(JSONL):
        return []
    return [json.loads(l) for l in open(JSONL) if l.strip()]

def append_row(row):
    with open(JSONL, "a") as f:
        f.write(json.dumps(row) + "\n")

done = {r["run_id"] for r in read_rows() if r.get("status") == "ok"}
print(len(done), "runs already recorded — these will be skipped")

In [ ]:
gate_lib = GateLibrary()

def run_one(scale_name, cfg, seed, arch):
    widths, steps = cfg["widths"], cfg["steps"]
    neurons = sum(widths)
    t_start = time.time()
    print(">>>", DATASET, scale_name, arch, "seed", seed, "|", neurons, "neurons", steps, "steps")

    gs = GroupSum(k=GROUPSUM_K, tau=GROUPSUM_TAU)
    topo = random_sparse(jax.random.PRNGKey(seed), INPUT_DIM, widths)
    t0 = time.time()
    if arch == "binary":
        net = BinaryDLGN(jax.random.PRNGKey(seed + 1), topo, init="randn")
        model, history = train_binary(net, optax.adam(BINARY_LR), bin_train_j, train_y_j,
                                      total_steps=steps, batch_size=BATCH_SIZE, group_sum=gs,
                                      log_every=max(steps // 100, 50))
        test_j, test_np = bin_test_j, bin_test
    else:
        net = PolynomialNetwork(jax.random.PRNGKey(seed + 2), topo)
        model, history = train(net, optax.adam(cfg["ternary_lr"]), ter_train_j, train_y_j,
                               total_steps=steps, batch_size=BATCH_SIZE, lambda_max=LAMBDA_MAX,
                               lambda_gamma=LAMBDA_GAMMA, loss_fn=make_ternary_loss(gs),
                               log_every=max(steps // 100, 50))
        test_j, test_np = ter_test_j, ter_test
    train_wall = time.time() - t0

    base = os.path.join(CKPT, DATASET, scale_name + "_" + arch + "_seed" + str(seed))
    with open(base + "_soft.pkl", "wb") as f:
        pickle.dump(model, f)
    with open(base + "_history.json", "w") as f:
        json.dump(history, f)

    t0 = time.time()
    if arch == "binary":
        hr = harden_binary_network(model); circuit = BinaryLearnedCircuit(hr)
    else:
        hr = harden_network_fast(model); circuit = TernaryLearnedCircuit(hr)
    harden_wall = time.time() - t0
    with open(base + "_harden.pkl", "wb") as f:
        pickle.dump(hr, f)

    t0 = time.time()
    soft_train = float(np.mean(evaluate_soft(model, bin_train_j if arch == "binary" else ter_train_j, gs) == train_y))
    soft_test = float(np.mean(evaluate_soft(model, test_j, gs) == test_y))
    circ_acc, unk = circuit_test_metrics(circuit, np.asarray(test_np, dtype=np.int8), test_y, N_CLASSES, arch)
    eval_wall = time.time() - t0

    print("    soft_test", round(soft_test * 100, 2), "circuit_test", round(circ_acc * 100, 2),
          "gap", round((soft_test - circ_acc) * 100, 2), "unknown", round(unk * 100, 2))

    append_row({
        "run_id": DATASET + "|" + scale_name + "|" + arch + "|seed" + str(seed),
        "dataset": DATASET, "scale": scale_name, "arch": ARCH_LABEL[arch], "seed": seed,
        "neurons": neurons, "layers": len(widths), "steps": steps,
        "soft_train_acc": soft_train, "soft_test_acc": soft_test, "circuit_test_acc": circ_acc,
        "unknown_pct": unk, "final_loss": float(history[-1]["total_loss"]),
        "unique_gates": len(hr.get("gate_census", {})),
        "train_wall_s": train_wall, "harden_wall_s": harden_wall, "eval_wall_s": eval_wall,
        "total_wall_s": time.time() - t_start,
        "steps_per_s": steps / train_wall if train_wall > 0 else 0.0,
        "lr": BINARY_LR if arch == "binary" else cfg["ternary_lr"],
        "batch_size": BATCH_SIZE, "gpu_name": GPU_NAME, "ckpt": base + "_soft.pkl", "status": "ok",
    })
    del model, net, circuit, hr
    gc.collect(); jax.clear_caches()

In [ ]:
for scale_name, cfg in SCALES.items():
    for seed in SEEDS:
        for arch in ["ternary", "binary"]:
            rid = DATASET + "|" + scale_name + "|" + arch + "|seed" + str(seed)
            if rid in done:
                print("skip", rid)
                continue
            run_one(scale_name, cfg, seed, arch)
print("done")

In [ ]:
rows = [r for r in read_rows() if r.get("status") == "ok" and r["dataset"] == DATASET]
order = list(FULL_SCALES) + ["smoke"]
rows.sort(key=lambda r: (order.index(r["scale"]) if r["scale"] in order else 99, r["arch"], r["seed"]))
print("scale          arch  seed  soft_test  circuit  gap(pp)  unknown%")
for r in rows:
    gap = (r["soft_test_acc"] - r["circuit_test_acc"]) * 100
    print(r["scale"].ljust(14), r["arch"].ljust(5), str(r["seed"]).ljust(5),
          str(round(r["soft_test_acc"] * 100, 2)).ljust(10),
          str(round(r["circuit_test_acc"] * 100, 2)).ljust(8),
          str(round(gap, 2)).ljust(8), round(r["unknown_pct"] * 100, 2))